# COMP5423 BrowseComp-Plus 智能体：系统实现与本地复测

本 Notebook 只围绕当前工作目标组织：**（1）系统实现；（2）本地复测**。代码实现位于项目 `src/` 和 `skills/`，Notebook 说明架构并提供本地演示、缓存/索引检查、验证集复测和输出检查。

模型 API key 只放在项目 `.env` 中。Notebook 的单题演示和批量复测会调用 DeepSeek API，产生费用；相关运行开关默认关闭。


## 1. 系统实现

### 1.1 系统目标与评分对应

系统在固定 BrowseComp-Plus 文档集合上回答多线索问题，结合检索、LLM 推理、工具调用和运行时加载的可复用技能。课程系统实现部分共 40%。

| 子项 | 分值 | 项目实现 |
|---|---:|---|
| 检索与工具设计 | 12% | BM25 稀疏检索及 BGE 密集检索；单一 `search` 检索工具；`open`、`find` 辅助工具 |
| 基础工具调用智能体 | 4% | DeepSeek V4 Flash function calling 选择工具并使用返回结果 |
| 迭代式智能体循环 | 5% | 后续搜索、打开、查找动作可依据前序观察调整 |
| 上下文管理 | 5% | 独立结构化状态、证据筛选和上下文裁剪 |
| 技能基本结构 | 6% | `skills/*/SKILL.md`，运行时加载 |
| 自动技能选择 bonus | 4% | 两个不同技能按问题特征或低证据状态自动选择 |
| 用户界面 | 4% | Streamlit 页面支持提问、查看答案、证据、技能和轨迹 |

检索方法类别评分为：一类 4 分、两类 6 分、三类 8 分；不同检索器必须通过**一个统一检索工具**暴露。当前实现有 BM25 和 BGE encoder dense；`dense_instruct` 只是 BGE 查询前缀，不能直接认定达到课程定义的 LLM 嵌入式指令检索第三类。工具评分要求统一检索工具加至少一种可调用辅助工具；报告需列出工具名称、输入输出和工具类型数。


## 2. 系统设计方法与关键组件

系统采用“检索器 + 有界工具调用智能体”。用户提交多线索问题后，DeepSeek V4 Flash 每轮选择一个结构化动作。BM25S 在固定语料上检索候选文档；智能体通过 `open` 读取文档全文，通过 `find` 定位短语，再基于打开过的证据给出答案和文档 ID 引用。检索分数、查询来源、打开状态、证据、轨迹和 token / 工具调用统计都会写入结果。

| 组件 | 作用 | 项目入口 |
|---|---|---|
| 数据加载 | 从 Hugging Face 数据集读取验证问题和固定语料；评测时可离线使用缓存 | `src/data.py` |
| 稀疏索引 | 在 100,195 篇文档上构建和载入 BM25S 索引 | `src/build_index.py`, `src/retrieval.py` |
| 可选向量检索 | BGE small embedding；支持 dense、instruction-aware dense 和 hybrid | `src/build_dense_index.py`, `src/dense.py` |
| 工具层 | 单查询 / 多查询检索、打开文档、短语查找、查询来源与排名记录 | `src/retrieval.py` |
| 智能体 | DeepSeek function calling、动作校验、技能选择、搜索恢复、上下文预算、证据引用与轨迹 | `src/agent.py`, `src/skills.py`, `skills/` |
| 评测与分析 | 生成预测 JSONL；计算 normalized EM、覆盖率、成本、停止原因及引用有效性 | `src/evaluate.py`, `src/analyze_results.py` |
| 用户界面 | Streamlit 交互式问题输入、答案、统计、轨迹和证据查看 | `src/app.py` |
| 本 Notebook | 将报告内容、历史实测结果和交互演示放在同一阅读 / 运行界面 | `COMP5423_SearchAgent.ipynb` |

API 密钥只从项目根目录的 `.env` 读取。预测文件和本 Notebook 不包含密钥内容。


## 3. 系统流程图

下图展示从输入问题到最终答案与界面输出的完整工作流。边界约束包括最大 agent 步数、搜索预算、动作重试数、检索候选数和上下文长度。

```text
┌─────────────────┐
│ 输入研究问题    │
└────────┬────────┘
         ▼
┌─────────────────────────┐
│ 选择相关推理技能 / 初始化状态 │
└────────┬────────────────┘
         ▼
┌─────────────────────────┐     ┌────────────────────────────┐
│ DeepSeek 选择一个结构化动作 │────▶│ 参数校验 / 无效动作恢复       │
└────────┬────────────────┘     └──────────────┬─────────────┘
         │ search / open / find                 │
         ▼                                      ▼
┌─────────────────────────┐     ┌────────────────────────────┐
│ BM25 / dense / hybrid 检索 │     │ 打开排名候选 / 检查短语        │
└────────┬────────────────┘     └──────────────┬─────────────┘
         └──────────────────┬───────────────────┘
                            ▼
                 ┌──────────────────────┐
                 │ 更新证据、历史与来源排名 │
                 └──────────┬───────────┘
                            ▼
                 ┌──────────────────────┐
                 │ 证据充分且达到停止条件？ │
                 └──────┬─────────┬─────┘
                      否│         │是 / 到达步数上限
                        └──回到模型│
                                   ▼
                 ┌──────────────────────┐
                 │ 答案 + 已打开文档引用   │
                 └──────────┬───────────┘
                            ▼
                 ┌──────────────────────┐
                 │ UI 展示答案、证据、轨迹 │
                 └──────────────────────┘
```


### 3.1 当前保留的系统参数

| 参数 | 当前值 | 说明 |
|---|---:|---|
| 模型 | `deepseek-v4-flash` | 通过 DeepSeek API function calling |
| 检索方式 | `bm25` | 当前完整端到端验证表现最佳 |
| 每次检索返回数 | 8 | top-k=8 |
| 多线索查询合并 | `best_rank` | 先按最佳排名排序，再用命中查询数打破平局 |
| 首次检索自动打开 | 开启，候选数 1 | 第一次成功搜索后打开最高排名文档 |
| agent 最大步数 | 8 | 最后一步保留给最终答案 |
| 搜索预算 | 3 | 单次搜索最多含 2 个额外线索查询 |
| 检索片段长度 | 700 字符 | 完整内容通过 `open` 获取 |
| 人物多跳引用门槛 | 2 个来源 | 引用必须来自实际打开且支持线索的文档 |



## 4. 系统实现与可靠性机制

- 使用 JSON schema 风格的 DeepSeek function calling，限定 `search`、`open`、`find`、`answer` 四种动作；每轮只执行一个工具动作。
- 在执行前检查动作参数、字段长度、引用格式和搜索状态；对格式错误的动作允许有限重试。
- 对重复查询、提前继续搜索、打开已读文档等状态错误，按已检索候选分数恢复到尚未检查的文档。
- 第一次成功检索后自动打开最高排名候选，减少模型未检查候选就继续搜索的情况。
- 限制上下文长度，并在裁剪时保留完整的工具调用 / 工具结果对，避免 API 对话协议不完整。
- 对搜索年份进行精度限制，避免模型凭空增加题目未给出的具体年份。
- 最终答案的引用必须来自已打开证据；对多跳人物识别要求至少两个不同来源。证据不足时允许弃答。
- 每个预测记录轨迹、请求 / 执行的工具调用、模型调用、token 数、重试次数和停止原因，方便追踪失败。

**实现代码入口：** [agent.py](src/agent.py)、[retrieval.py](src/retrieval.py)、[data.py](src/data.py)、[app.py](src/app.py)。


### 1.6 技能与自动选择

项目包含两个不同用途的技能：

- `multi_hop_identity`：拆解人物/实体多线索问题，并要求独立证据核验。
- `search_recovery`：搜索无有效结果或证据不足时，围绕未解决线索改写查询。

技能在运行时加载，不永久写入基础提示词。自动选择依据问题是否请求人物身份，以及搜索状态是否为空或证据不足。课程 bonus 还要求报告和演示视频展示不同问题触发不同技能；下方交互单元格可查看实际 `selected_skills`。


## 7. 用户界面设计

Streamlit 主界面让用户在 **Research question** 文本框输入问题并点击 **Search**。页面展示最终答案、支持证据的文档 ID、工具调用数、token 数、技能选择、动作重试次数和模型调用数；两个展开区分别显示 agent trajectory 和 collected evidence。Notebook 后面的演示单元格则适合逐步执行和检查每个阶段。

### 界面截图

本 Notebook 当前项目目录没有保存 Streamlit 界面截图，因此这里不放伪造截图。启动界面的步骤见附录 B；打开本地界面后可用系统截图工具截取主界面和一次问答结果，再将图片保存为 `report/streamlit_ui_screenshot.png`。下方单元格会在文件存在时自动显示它。


In [ ]:
from pathlib import Path
from IPython.display import display, Image, Markdown
ui_shot = Path("report/streamlit_ui_screenshot.png")
if ui_shot.exists():
    display(Image(filename=str(ui_shot)))
else:
    display(Markdown("**截图待补：** `report/streamlit_ui_screenshot.png` 尚不存在。运行 Streamlit 后截取实际界面并保存到该路径。"))


### 1.8 单题交互演示

从下方单元格输入一个问题，按顺序运行后续单元格。运行会调用模型 API。若只需检查 Notebook，可保持问题为空，避免 API 请求。


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "agent.py").exists():
    raise RuntimeError(
        f"当前目录不是 COMP5423 项目根目录：{PROJECT_ROOT}。"
        "请从 D:/COMP5423 启动 Jupyter，或在 VS Code 中打开该项目。"
    )
os.chdir(PROJECT_ROOT)

# 数据与 BM25 索引已在本机缓存；运行时不再请求 Hugging Face。
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

print(f"Project: {PROJECT_ROOT}")
print(f"Python:  {sys.executable}")
print(f".env:    {'found' if (PROJECT_ROOT / '.env').is_file() else 'MISSING'}")


In [ ]:
import bm25s
from pathlib import Path
import sys

EXPECTED_PYTHON = (PROJECT_ROOT / ".venv" / "Scripts" / "python.exe").resolve()
ACTIVE_PYTHON = Path(sys.executable).resolve()
if ACTIVE_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Notebook 当前使用的 Python 是 {ACTIVE_PYTHON}，而项目依赖安装在 {EXPECTED_PYTHON}。"
        "请点 VS Code 右上角的 Kernel 选择器，选择 `Python (.venv COMP5423)`，然后重新运行本单元格。"
    )
from IPython.display import display, Markdown, JSON

from src.config import settings
from src.data import load_corpus
from src.retrieval import create_search_tools
from src.agent import SearchAgent

if not (PROJECT_ROOT / ".env").is_file():
    raise RuntimeError("找不到项目 .env。请先复制 .env.example 为 .env 并配置 DeepSeek API key。")
if not settings.index_dir.exists():
    raise FileNotFoundError(
        f"BM25 索引不存在：{settings.index_dir}。先运行 `python -m src.build_index`。"
    )

print("加载 D 盘本地语料缓存……")
corpus = load_corpus(local_files_only=True)
print(f"已加载 {len(corpus):,} 篇文档。")

print("加载已构建的 BM25 索引……")
bm25_index = bm25s.BM25.load(str(settings.index_dir), mmap=True)
search_tools = create_search_tools(corpus, bm25_index, top_k=settings.top_k)
agent = SearchAgent(search_tools)
print(f"智能体已就绪：model={settings.model}, retrieval={settings.retrieval_mode}, top_k={settings.top_k}")


In [ ]:
# 留空时跳过交互演示，避免“全部运行”时意外调用 API。需要演示时再填写问题。
question = ""
question = question.strip()
if question:
    print("当前问题：", question)
else:
    print("交互演示暂不运行。填写 question 后，单独运行本单元格及后续演示单元格。")


In [ ]:
if question:
    result = agent.run(question)
    print("运行完成")
else:
    result = None
    print("已跳过 API 调用。")


In [ ]:
if result is not None:
    final_action = next(
        (step.get("action", {}) for step in reversed(result["trajectory"])
         if isinstance(step.get("action"), dict) and step["action"].get("action") == "answer"),
        {},
    )
    citation_ids = final_action.get("supporting_doc_ids", [])

    display(Markdown("### 答案\n" + (result.get("answer") or "（模型未返回答案）")))
    print("引用文档：", ", ".join(citation_ids) if citation_ids else "无（模型选择弃答或未提供引用）")
    print({
        "tool_calls": result.get("tool_calls"),
        "model_calls": result.get("llm_calls"),
        "total_tokens": result.get("total_tokens"),
        "invalid_action_retries": result.get("invalid_action_retries"),
        "stop_reason": result.get("stop_reason"),
        "selected_skills": result.get("selected_skills", []),
    })


In [ ]:
if result is not None:
    evidence_by_id = {str(item["doc_id"]): item for item in result["state"].get("evidence", [])}
    for doc_id in citation_ids:
        item = evidence_by_id.get(str(doc_id))
        if item is None:
            print(f"引用 {doc_id} 不在已收集证据中。")
            continue
        display(Markdown(f"### 文档 {doc_id}\n来源：{item.get('url', '')}"))
        display(Markdown(str(item.get("text", ""))[:3000]))

    if not citation_ids:
        print("本次没有引用文档。可在下方轨迹中检查智能体为何弃答。")


In [ ]:
if result is not None:
    display(JSON(result["trajectory"]))


## 2. 本地复测

### 2.1 复测范围与当前基线

本地复测先用带标签的 100 题 validation 检查端到端运行和 normalized EM，再按需要跑单题 smoke test。历史保留配置的完整 validation 结果为：100/100 预测覆盖、23/100 normalized EM、平均 6.47 次工具调用、平均 24,984.45 tokens、65/65 引用有效。此结果是开发验证基线，不代表最终测试集排名。

复测分为：环境与缓存预检 →（首次运行时）下载数据/构建索引 → 少量 smoke test → 全量 validation → JSONL 结构和指标检查。全量复测须主动开启开关。


In [ ]:
DOWNLOAD_CORPUS = False
BUILD_INDEX = False

if DOWNLOAD_CORPUS:
    # 允许 Hugging Face 下载；完成后，标准评测命令仍使用本地缓存。
    os.environ["HF_HUB_OFFLINE"] = "0"
    os.environ["HF_DATASETS_OFFLINE"] = "0"
    from src.data import load_corpus
    downloaded_corpus = load_corpus(local_files_only=False)
    print(f"语料已缓存：{len(downloaded_corpus):,} 篇文档")

if BUILD_INDEX:
    import subprocess
    completed = subprocess.run(
        [sys.executable, "-m", "src.build_index"], cwd=PROJECT_ROOT, check=False
    )
    if completed.returncode != 0:
        raise RuntimeError("索引构建失败，请检查上方输出。")
    print("BM25 索引构建完成。")

if not DOWNLOAD_CORPUS and not BUILD_INDEX:
    print(f"使用本地缓存：{settings.data_cache_dir}")
    print(f"BM25 索引：{settings.index_dir} ({'已存在' if settings.index_dir.exists() else '缺失'})")


### 2.2 环境、缓存与索引预检

In [ ]:
from pathlib import Path
import sys

expected_python = (PROJECT_ROOT / ".venv" / "Scripts" / "python.exe").resolve()
active_python = Path(sys.executable).resolve()
checks = {
    "correct_project_root": (PROJECT_ROOT / "src" / "evaluate.py").is_file(),
    "project_virtualenv_kernel": active_python == expected_python,
    "local_env_file_exists": (PROJECT_ROOT / ".env").is_file(),
    "corpus_cache_directory_exists": settings.data_cache_dir.exists(),
    "bm25_index_exists": settings.index_dir.exists(),
}
print("Environment checks:", checks)
if not all(checks.values()):
    raise RuntimeError("请先修复上述失败项；不要在 Notebook 中粘贴 API key。")


### 2.3 少量 smoke test 与全量 validation 复测

`EVALUATION_LIMIT=1` 可先做单题 smoke test；设为 `100` 或 `None` 可覆盖完整 validation。运行前确认 `.env` 中 API key 有效，并考虑 API 费用。输出文件会保存完整轨迹和调用统计。


In [ ]:
import subprocess

RUN_VALIDATION_RETEST = False  # 确认后改为 True；默认不会调用 API
EVALUATION_LIMIT = 100        # standardized full validation
EVALUATION_OFFSET = 0
RETEST_OUTPUT = PROJECT_ROOT / "results" / "notebook_local_retest_full100_repeat.jsonl"

if RUN_VALIDATION_RETEST:
    cmd = [sys.executable, "-m", "src.evaluate", "--split", "validation",
           "--offset", str(EVALUATION_OFFSET), "--output", str(RETEST_OUTPUT)]
    if EVALUATION_LIMIT is not None:
        cmd.extend(["--limit", str(EVALUATION_LIMIT)])
    proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, text=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, encoding="utf-8", errors="replace", bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"validation retest failed, exit code={code}")
else:
    print("复测未启动。确认费用后设置 RUN_VALIDATION_RETEST=True，再运行此单元格。")


### 2.4 检查预测文件并计算验证指标

In [ ]:
import json

if RUN_VALIDATION_RETEST and RETEST_OUTPUT.exists():
    rows = [json.loads(line) for line in RETEST_OUTPUT.read_text(encoding="utf-8").splitlines() if line.strip()]
    required = {"id", "question", "answer", "trajectory", "tool_calls", "total_tokens"}
    missing = sorted({key for row in rows for key in required - set(row)})
    ids = [str(row.get("id", "")) for row in rows]
    unique_ids = len(ids) == len(set(ids))
    trajectory_lists = all(isinstance(row.get("trajectory"), list) for row in rows)
    print({"rows":len(rows), "missing_fields":missing, "unique_ids":unique_ids,
           "trajectory_is_list":trajectory_lists, "output":str(RETEST_OUTPUT)})
    if missing or not unique_ids or not trajectory_lists:
        raise ValueError("预测文件结构检查失败。")
    analysis = subprocess.run([sys.executable,"-m","src.analyze_results",str(RETEST_OUTPUT)],
                              cwd=PROJECT_ROOT,check=True,text=True,capture_output=True)
    print(analysis.stdout)
else:
    print("暂无本次复测输出；完成 2.3 后再运行本单元格。")


### 2.5 本地复测判定标准

- Smoke test：进程正常结束，至少写出一条预测，必需字段齐全，轨迹为机器可读列表。
- 完整 validation：预期 100 条且 ID 不重复；analyzer 输出 normalized EM、覆盖率、工具调用、token、停止原因和引用诊断。
- 结果复核：对照 `results/notebook_local_retest.jsonl` 与 sidecar 元数据，保留实际跑出的数据，不手工修改预测或轨迹。
- 更改配置后：先跑相同 offset 的小样本作配对比较；若值得保留，再重新跑完整 validation。

> 本地复测不等于最终隐藏测试评测。当前阶段聚焦系统实现和可复现本地复测；test split 的官方提交另按课程发布的数据运行。


## 3. 优化实验 B：降低多跳人物题的来源数门槛

本文件是独立的下一轮优化实验，保留前两节的系统实现和本地复测流程。原版 notebook 与上一版优化 notebook、已保存的 25/100 最佳候选结果都不会被覆盖。

实验假设：当前严格答案格式全量候选在 100 题中弃答 65 题；把多跳人物题最低引用来源从 2 个调到 1 个，允许一份内容确实覆盖多条线索的已打开文档作为支持，可能减少证据充分时的过度弃答。系统仍要求引用已打开来源；如果来源没有明确覆盖线索，模型不应据此猜测。此实验只先跑 20 题 pilot，比较同一题号范围，不直接覆盖 25/100 最佳候选。

### 3.1 当前保留的最佳候选

| 结果 | 验证集 EM | 弃答 | 非弃答错误 | 引用情况 |
|---|---:|---:|---:|---|
| 历史旧版 `validation_action_fix_100_v3` | **36/100** | 44 | 20 | 未输出引用 ID，缺少运行元数据 |
| 原始可引用基线 | 21/100 | 70 | 9 | 57/57 有效 |
| 严格答案格式 | 25/100 | 65 | 10 | 62/65 有效 |
| 严格格式 + 最低 1 个身份来源 | 33/100 | **59** | **8** | **61/61 有效** |

若只按 normalized EM 排序，历史旧版 36/100 是最高值；但它没有引用 ID，且非弃答错误较多。当前完整追踪引用、错误更少且成本较低的最佳配置是 33/100。不要把这两种标准混称为同一个“最佳”。旧文件与所有新文件均保留。

### 3.2 最低来源数配置：全量复测（100 题）

本实验的 pilot 已完成，且该配置已有 33/100 全量结果。以下单元统一按 offset=0、limit=100 复测，使用新的输出文件，不覆盖当前最佳结果。默认关闭；只有需要复现时才打开开关。

In [ ]:
import os
import subprocess
import sys

RUN_FULL_MIN_SOURCES1_RERUN = False
MIN_SOURCES1_RERUN_OUTPUT = PROJECT_ROOT / "results" / "optimization_min_sources1_strict_validation100_rerun.jsonl"
if RUN_FULL_MIN_SOURCES1_RERUN:
    env = os.environ.copy()
    env["STRICT_FINAL_ANSWER_FORMAT"] = "true"
    env["MIN_IDENTITY_SOURCES"] = "1"
    env["HF_HUB_OFFLINE"] = "1"
    env["HF_DATASETS_OFFLINE"] = "1"
    cmd = [sys.executable, "-m", "src.evaluate", "--split", "validation", "--offset", "0",
           "--limit", "100", "--output", str(MIN_SOURCES1_RERUN_OUTPUT)]
    subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, check=True)
    subprocess.run([sys.executable, "-m", "src.analyze_results", str(MIN_SOURCES1_RERUN_OUTPUT)],
                   cwd=PROJECT_ROOT, env=env, check=True)
else:
    print("Full validation rerun is disabled; the retained 33/100 result remains unchanged.")


### 3.5 全量结果：最低来源数为 1（2026-11-14）

全量文件：`results/optimization_min_sources1_strict_validation100.jsonl`；元数据确认 validation 100/100、`strict_final_answer_format=true`、`min_identity_sources=1`。

| 指标 | 最低 2 个来源（25/100 候选） | 最低 1 个来源（本次） |
|---|---:|---:|
| Normalized EM | 25/100 | **33/100** |
| 弃答 | 65 | **59** |
| 错误的非弃答 | 10 | **8** |
| 引用有效 | 62/65 (95.38%) | **61/61 (100%)** |
| 平均工具调用 | 6.50 | **6.20** |
| 平均 tokens | 24,994.96 | **23,629.88** |

逐题配对中，新配置独有答对 16 题，旧配置独有答对 8 题；相对原始基线，新配置净增 12 题。该结果支持把它保留为当前最佳配置。API 输出仍有随机性，后续新实验应继续保留独立输出文件并用同一 100 题做对照。

## 4. 优化实验 C：自动打开两个候选文档（历史全量结果）

该方案已在 validation 全量 100 题评测，27/100，低于当前 top-1 配置的 33/100。它仅作为历史对照保留；若要复现，下面的单元也使用 offset=0、limit=100 和独立输出文件。

### 4.1 top-2 全量复测（100 题，默认关闭）

已有全量结果，不需要为了 Run All 重复调用 API。若要复现，请手动打开开关；输出会写入独立文件。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

RUN_AUTO_OPEN2_FULL_RERUN = False  # completed; full 100-question rerun is opt-in
AUTO_OPEN2_OFFSET = 0
AUTO_OPEN2_LIMIT = 100
AUTO_OPEN2_OUTPUT = PROJECT_ROOT / "results" / "optimization_min_sources1_strict_autoopen2_validation100_rerun.jsonl"

if RUN_AUTO_OPEN2_FULL_RERUN:
    env = os.environ.copy()
    env["STRICT_FINAL_ANSWER_FORMAT"] = "true"
    env["MIN_IDENTITY_SOURCES"] = "1"
    env["AUTO_OPEN_FIRST_CANDIDATE"] = "true"
    env["AUTO_OPEN_CANDIDATE_COUNT"] = "2"
    env["HF_HUB_OFFLINE"] = "1"
    env["HF_DATASETS_OFFLINE"] = "1"
    cmd = [sys.executable, "-m", "src.evaluate", "--split", "validation",
           "--offset", str(AUTO_OPEN2_OFFSET), "--limit", str(AUTO_OPEN2_LIMIT),
           "--output", str(AUTO_OPEN2_OUTPUT)]
    proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            encoding="utf-8", errors="replace", bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError("Auto-open top-2 full validation rerun failed.")
else:
    print("Full top-2 validation rerun disabled. Set RUN_AUTO_OPEN2_FULL_RERUN=True to run all 100 questions.")


### 4.2 与当前最佳配置全量逐题对比

如上方复现单元已运行，则将 top-2 结果与当前最佳完整 100 题按 question ID 对齐，比较 EM、弃答、错误非弃答、引用和成本。历史全量结果已记录在 4.3。

In [ ]:
import json
from src.data import load_validation
from src.analyze_results import normalize, record_diagnostics

AUTO_OPEN2_CONTROL = PROJECT_ROOT / "results" / "optimization_min_sources1_strict_validation100.jsonl"
if RUN_AUTO_OPEN2_FULL_RERUN and AUTO_OPEN2_OUTPUT.exists() and AUTO_OPEN2_CONTROL.exists():
    validation = {str(r["id"]): r for r in load_validation("validation", local_files_only=True)}
    def read_autoopen_rows(path):
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    def summarize_autoopen(rows):
        n = len(rows)
        diagnostics = [record_diagnostics(r) for r in rows]
        correct = sum(normalize(r["answer"]) == normalize(validation[str(r["id"])]["answer"]) for r in rows)
        citation_count = sum(len(d["citations"]) for d in diagnostics)
        invalid_citations = sum(d["invalid_citation_count"] for d in diagnostics)
        return {"n": n, "correct": f"{correct}/{n}", "abstentions": sum(d["abstained"] for d in diagnostics),
                "wrong_non_abstaining": sum(not d["abstained"] and normalize(r["answer"]) != normalize(validation[str(r["id"])]["answer"]) for r, d in zip(rows, diagnostics)),
                "citation_validity": round((citation_count-invalid_citations)/citation_count, 3) if citation_count else None,
                "mean_tool_calls": round(sum(r.get("tool_calls", 0) for r in rows)/max(n,1), 2),
                "mean_tokens": round(sum(r.get("total_tokens", 0) for r in rows)/max(n,1), 1)}
    control_rows = read_autoopen_rows(AUTO_OPEN2_CONTROL)[AUTO_OPEN2_OFFSET:AUTO_OPEN2_OFFSET+AUTO_OPEN2_LIMIT]
    trial_rows = read_autoopen_rows(AUTO_OPEN2_OUTPUT)
    if [str(r["id"]) for r in control_rows] != [str(r["id"]) for r in trial_rows]:
        raise ValueError("Control and auto-open top-2 pilot IDs do not match.")
    print("current_best_top1:", summarize_autoopen(control_rows))
    print("trial_top2:", summarize_autoopen(trial_rows))
else:
    print("Run the auto-open top-2 pilot and retain the current best result file before comparing.")


### 4.3 top-2 全量结果记录（2026-11-14）

top-2 的全量评测都低于当前 top-1 最佳配置 33/100，故不升级：

| top-2 全量运行 | EM | 弃答 | 错误非弃答 | 引用 ID 有效 | 平均调用 | 平均 tokens |
|---|---:|---:|---:|---:|---:|---:|
| 初次全量 | 27/100 | 65 | 10 | 51/51 | 7.55 | 25,057.36 |
| 重跑 | 27/100 | 60 | 13 | 60/60 | 7.38 | 23,633.21 |

结果文件分别为 `results/optimization_min_sources1_strict_autoopen2_validation100.jsonl` 与 `results/optimization_min_sources1_strict_autoopen2_validation100_rerun2.jsonl`。原始文件名曾误写为 pilot，现已整理为 validation100 命名。

## 5. 优化路径 1：按独立查询覆盖度重排候选

问题拆解和证据表需要改变 agent 的规划行为，较难单独归因。本轮先实现更小、可控的结构改动：当 agent 在一个 `search` action 中提交主查询与多个独立线索查询时，计算每篇文档的多查询命中数；在原 RRF 相关性分数上增加小幅覆盖奖励，再选 top-k 候选。只对跨查询命中的候选加分，仍必须 `open` 文档并根据正文核实；查询命中数不视为事实证据。

固定当前最佳的答案格式、最低身份来源数和自动打开 top-1；仅将 `QUERY_FUSION_MODE` 切换为 `clue_coverage`。全量运行 validation offset 0、limit 100，对照当前最佳 full-100 的相同题号切片。

### 5.1 运行 100 题全量验证

默认直接运行。请使用项目 `.venv` kernel 并从上到下运行 notebook。输出到独立文件，不覆盖 top-1 当前最佳全量结果。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

RUN_CLUE_COVERAGE_VALIDATION = False  # completed; enable only for a deliberate full rerun
CLUE_COVERAGE_OFFSET = 0
CLUE_COVERAGE_LIMIT = 100
CLUE_COVERAGE_WEIGHT = 0.02
CLUE_COVERAGE_OUTPUT = PROJECT_ROOT / "results" / "optimization_clue_coverage_min1_validation100.jsonl"

if RUN_CLUE_COVERAGE_VALIDATION:
    env = os.environ.copy()
    env["STRICT_FINAL_ANSWER_FORMAT"] = "true"
    env["MIN_IDENTITY_SOURCES"] = "1"
    env["AUTO_OPEN_FIRST_CANDIDATE"] = "true"
    env["AUTO_OPEN_CANDIDATE_COUNT"] = "1"
    env["QUERY_FUSION_MODE"] = "clue_coverage"
    env["QUERY_COVERAGE_WEIGHT"] = str(CLUE_COVERAGE_WEIGHT)
    env["HF_HUB_OFFLINE"] = "1"
    env["HF_DATASETS_OFFLINE"] = "1"
    cmd = [sys.executable, "-m", "src.evaluate", "--split", "validation",
           "--offset", str(CLUE_COVERAGE_OFFSET), "--limit", str(CLUE_COVERAGE_LIMIT),
           "--output", str(CLUE_COVERAGE_OUTPUT)]
    proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            encoding="utf-8", errors="replace", bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError("Clue-coverage full validation failed.")
else:
    print("Full validation disabled. Set RUN_CLUE_COVERAGE_VALIDATION=True to run all 100 questions.")


### 5.2 全量逐题对比

对照使用当前最佳 full-100 结果的题号 0–99。除 EM 外也看弃答、非弃答错误、引用 ID 有效率和成本；若没有准确数提升，或引用/错误率恶化，就保留原 `best_rank` 策略。

In [ ]:
import json
from src.data import load_validation
from src.analyze_results import normalize, record_diagnostics

CLUE_COVERAGE_CONTROL = PROJECT_ROOT / "results" / "optimization_min_sources1_strict_validation100.jsonl"
if RUN_CLUE_COVERAGE_VALIDATION and CLUE_COVERAGE_OUTPUT.exists() and CLUE_COVERAGE_CONTROL.exists():
    validation = {str(r["id"]): r for r in load_validation("validation", local_files_only=True)}
    def read_coverage_rows(path):
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    def summarize_coverage(rows):
        n = len(rows)
        diagnostics = [record_diagnostics(r) for r in rows]
        correct = sum(normalize(r["answer"]) == normalize(validation[str(r["id"])]["answer"]) for r in rows)
        cited = sum(len(d["citations"]) for d in diagnostics)
        invalid = sum(d["invalid_citation_count"] for d in diagnostics)
        return {"n": n, "correct": f"{correct}/{n}", "abstentions": sum(d["abstained"] for d in diagnostics),
                "wrong_non_abstaining": sum(not d["abstained"] and normalize(r["answer"]) != normalize(validation[str(r["id"])]["answer"]) for r, d in zip(rows, diagnostics)),
                "citation_id_validity": round((cited-invalid)/cited, 3) if cited else None,
                "mean_tool_calls": round(sum(r.get("tool_calls", 0) for r in rows)/max(n,1), 2),
                "mean_tokens": round(sum(r.get("total_tokens", 0) for r in rows)/max(n,1), 1)}
    control = read_coverage_rows(CLUE_COVERAGE_CONTROL)[CLUE_COVERAGE_OFFSET:CLUE_COVERAGE_OFFSET+CLUE_COVERAGE_LIMIT]
    trial = read_coverage_rows(CLUE_COVERAGE_OUTPUT)
    if [str(r["id"]) for r in control] != [str(r["id"]) for r in trial]:
        raise ValueError("Control and clue-coverage full-validation IDs do not match.")
    print("current_best_best_rank:", summarize_coverage(control))
    print("trial_clue_coverage:", summarize_coverage(trial))
else:
    print("Run the clue-coverage full validation and keep the current best result in results/ before comparing.")


### 5.3 线索覆盖全量结果（2026-11-14）

全量输出：`results/optimization_clue_coverage_min1_validation100.jsonl`；元数据确认 100/100、offset=0、`QUERY_FUSION_MODE=clue_coverage`、coverage weight=0.02。

| 指标 | 当前最佳：best_rank | clue_coverage |
|---|---:|---:|
| Normalized EM | **33/100** | 27/100 |
| 弃答 | 59 | 61 |
| 错误的非弃答 | **8** | 12 |
| 引用 ID 有效性 | 61/61 (100%) | 57/57 (100%) |
| 平均工具调用 | **6.20** | 6.61 |
| 平均 tokens | **23,629.88** | 25,402.47 |

逐题配对：clue_coverage 独有答对 6 题，best_rank 独有答对 12 题。该路线在本次全量测试中未提升，保留为对照，不替换当前最佳；评测开关默认关闭，避免误重复调用 API。

## 6. 优化路线 2：显式候选状态与重复动作恢复

本实验只改 agent 的候选状态反馈，不改检索排序、top-k、来源门槛或自动打开数量。开启 `EXPLICIT_CANDIDATE_STATE` 后，状态消息会显式列出未打开候选 ID；重复打开时错误说明会指出文档已在证据表、建议选择哪些未打开 ID；自动恢复打开候选后也会把失败原因和下一步提示写入 observation。

目标是减少重复 `open` 请求和备用打开事件，同时避免丢失尚未检查的候选。与当前最佳配置在同一 validation 100 题上逐题对比，使用独立输出文件。

### 6.1 全量回测（100 题）

默认开启，使用 `offset=0, limit=100`。Run All 时，前面的已完成实验均关闭，本单元执行这一次完整回测。

In [ ]:
import os
import subprocess
import sys

RUN_EXPLICIT_STATE_VALIDATION = False  # completed; enable only for a deliberate full rerun
EXPLICIT_STATE_OFFSET = 0
EXPLICIT_STATE_LIMIT = 100
EXPLICIT_STATE_OUTPUT = PROJECT_ROOT / "results" / "optimization_explicit_open_state_min1_validation100.jsonl"

if RUN_EXPLICIT_STATE_VALIDATION:
    env = os.environ.copy()
    env["STRICT_FINAL_ANSWER_FORMAT"] = "true"
    env["MIN_IDENTITY_SOURCES"] = "1"
    env["AUTO_OPEN_FIRST_CANDIDATE"] = "true"
    env["AUTO_OPEN_CANDIDATE_COUNT"] = "1"
    env["QUERY_FUSION_MODE"] = "best_rank"
    env["EXPLICIT_CANDIDATE_STATE"] = "true"
    env["HF_HUB_OFFLINE"] = "1"
    env["HF_DATASETS_OFFLINE"] = "1"
    cmd = [sys.executable, "-m", "src.evaluate", "--split", "validation",
           "--offset", str(EXPLICIT_STATE_OFFSET), "--limit", str(EXPLICIT_STATE_LIMIT),
           "--output", str(EXPLICIT_STATE_OUTPUT)]
    proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            encoding="utf-8", errors="replace", bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError("Explicit candidate-state validation failed.")
else:
    print("Full validation disabled. Set RUN_EXPLICIT_STATE_VALIDATION=True to run all 100 questions.")


### 6.2 与当前最佳配置全量对比

对比 normalized EM、弃答、错误的非弃答、引用 ID 有效率、平均调用/token、重复打开 fallback 数和被忽略的额外工具调用数。两份预测必须覆盖并按相同 100 个 question ID 排列。

In [ ]:
import json
from src.data import load_validation
from src.analyze_results import normalize, record_diagnostics

EXPLICIT_STATE_CONTROL = PROJECT_ROOT / "results" / "optimization_min_sources1_strict_validation100.jsonl"
if RUN_EXPLICIT_STATE_VALIDATION and EXPLICIT_STATE_OUTPUT.exists() and EXPLICIT_STATE_CONTROL.exists():
    validation = {str(r["id"]): r for r in load_validation("validation", local_files_only=True)}
    def read_state_rows(path):
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    def summarize_state(rows):
        n = len(rows)
        diagnostics = [record_diagnostics(r) for r in rows]
        correct = sum(normalize(r["answer"]) == normalize(validation[str(r["id"])]["answer"]) for r in rows)
        cited = sum(len(d["citations"]) for d in diagnostics)
        invalid = sum(d["invalid_citation_count"] for d in diagnostics)
        extras = sum(sum(1 for e in r.get("trajectory", []) if e.get("event") == "extra_function_calls_ignored") for r in rows)
        return {"n": n, "correct": f"{correct}/{n}", "abstentions": sum(d["abstained"] for d in diagnostics),
                "wrong_non_abstaining": sum(not d["abstained"] and normalize(r["answer"]) != normalize(validation[str(r["id"])]["answer"]) for r, d in zip(rows, diagnostics)),
                "citation_id_validity": round((cited-invalid)/cited, 3) if cited else None,
                "mean_tool_calls": round(sum(r.get("tool_calls", 0) for r in rows)/max(n,1), 2),
                "mean_tokens": round(sum(r.get("total_tokens", 0) for r in rows)/max(n,1), 1),
                "duplicate_open_fallbacks": sum(d["open_action_fallback_opens"] for d in diagnostics),
                "ignored_extra_tool_calls": extras,
                "invalid_action_retries": sum(r.get("invalid_action_retries", 0) for r in rows)}
    control = read_state_rows(EXPLICIT_STATE_CONTROL)
    trial = read_state_rows(EXPLICIT_STATE_OUTPUT)
    if len(control) != 100 or len(trial) != 100 or [str(r["id"]) for r in control] != [str(r["id"]) for r in trial]:
        raise ValueError("Both full-validation files must contain the same ordered 100 question IDs.")
    print("current_best:", summarize_state(control))
    print("explicit_state:", summarize_state(trial))
else:
    print("Run the explicit-state full validation and keep the current best result before comparing.")


### 6.3 显式候选状态全量结果（2026-11-14）

全量结果：`results/optimization_explicit_open_state_min1_validation100.jsonl`；元数据确认 validation 100/100、offset=0、`explicit_candidate_state=true`。

| 指标 | 当前最佳 | 显式状态实验 |
|---|---:|---:|
| Normalized EM | **33/100** | 11/100 |
| 弃答 | **59** | 82 |
| 错误的非弃答 | 8 | 7 |
| 引用 ID 有效性 | 61/61 (100%) | 30/30 (100%) |
| 平均工具调用 | **6.20** | 7.79 |
| 平均 tokens | **23,629.88** | 26,890.11 |
| 重复打开 fallback | 99 | **28** |
| 多工具调用事件 | 45 | 238 |

逐题配对中新配置独有答对 1 题，当前最佳独有答对 23 题。该改动减少了重复打开，但总体准确率明显下降、弃答和调用成本升高，因此不采纳；保留结果作消融对照，并关闭自动重跑。

## 7. 优化路线 3：可核验的结构化证据表与作答门槛

当前 agent 已有“维护证据账本”的文字提示，但并没有结构化输出或程序核验。本实验把最终 answer action 扩展为 `evidence_ledger=[{clue, doc_id, quote}]`：每条记录将一个问题线索映射到已打开文档及原文摘录；程序检查文档已打开、摘录能在该文档正文中找到、账本文档均包含在引用中。多跳人物题至少需要两条不同线索记录；同一文档只有在摘录实际支持线索时才可重复支撑。

只开启 `STRUCTURED_EVIDENCE_LEDGER`，其他参数固定为当前最佳（strict format、min identity sources=1、best_rank、auto-open top-1）。完整验证集全量运行，结果写入新文件。

### 7.1 全量回测（validation 100 题）

默认开启，offset=0、limit=100。此前已完成的 baseline、路径 1、路径 2 和 top-2 重跑开关均保持关闭，因此 Run All 只会执行本实验的这一次全量 API 回测。

In [ ]:
import os
import subprocess
import sys

RUN_STRUCTURED_EVIDENCE_VALIDATION = True
STRUCTURED_EVIDENCE_OFFSET = 0
STRUCTURED_EVIDENCE_LIMIT = 100
STRUCTURED_EVIDENCE_OUTPUT = PROJECT_ROOT / "results" / "optimization_structured_evidence_min1_validation100.jsonl"

if RUN_STRUCTURED_EVIDENCE_VALIDATION:
    env = os.environ.copy()
    env["STRICT_FINAL_ANSWER_FORMAT"] = "true"
    env["MIN_IDENTITY_SOURCES"] = "1"
    env["AUTO_OPEN_FIRST_CANDIDATE"] = "true"
    env["AUTO_OPEN_CANDIDATE_COUNT"] = "1"
    env["QUERY_FUSION_MODE"] = "best_rank"
    env["EXPLICIT_CANDIDATE_STATE"] = "false"
    env["STRUCTURED_EVIDENCE_LEDGER"] = "true"
    env["HF_HUB_OFFLINE"] = "1"
    env["HF_DATASETS_OFFLINE"] = "1"
    cmd = [sys.executable, "-m", "src.evaluate", "--split", "validation",
           "--offset", str(STRUCTURED_EVIDENCE_OFFSET), "--limit", str(STRUCTURED_EVIDENCE_LIMIT),
           "--output", str(STRUCTURED_EVIDENCE_OUTPUT)]
    proc = subprocess.Popen(cmd, cwd=PROJECT_ROOT, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            encoding="utf-8", errors="replace", bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError("Structured-evidence full validation failed.")
else:
    print("Full validation disabled. Set RUN_STRUCTURED_EVIDENCE_VALIDATION=True to run all 100 questions.")


### 7.2 与当前最佳全量对比

比较 EM、弃答、错误非弃答、引用 ID 有效率、平均调用与 tokens、evidence ledger 覆盖率和无效动作重试。两份结果必须包含完全相同的 100 个 validation ID。

In [ ]:
import json
from src.data import load_validation
from src.analyze_results import normalize, record_diagnostics

STRUCTURED_EVIDENCE_CONTROL = PROJECT_ROOT / "results" / "optimization_min_sources1_strict_validation100.jsonl"
if RUN_STRUCTURED_EVIDENCE_VALIDATION and STRUCTURED_EVIDENCE_OUTPUT.exists() and STRUCTURED_EVIDENCE_CONTROL.exists():
    validation = {str(r["id"]): r for r in load_validation("validation", local_files_only=True)}
    def read_evidence_rows(path):
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    def summarize_evidence(rows):
        n = len(rows)
        diagnostics = [record_diagnostics(r) for r in rows]
        correct = sum(normalize(r["answer"]) == normalize(validation[str(r["id"])]["answer"]) for r in rows)
        cited = sum(len(d["citations"]) for d in diagnostics)
        invalid = sum(d["invalid_citation_count"] for d in diagnostics)
        final_actions = [next((s.get("action", {}) for s in reversed(r.get("trajectory", [])) if isinstance(s, dict) and isinstance(s.get("action"), dict) and s["action"].get("action") == "answer" and not (isinstance(s.get("observation"), dict) and "error" in s["observation"])), {}) for r in rows]
        answered_actions = [a for a in final_actions if a and not any(m in a.get("answer", "").casefold() for m in ("insufficient evidence", "cannot verify", "could not verify", "unable to identify", "not enough information", "cannot determine"))]
        ledger_answers = sum(bool(a.get("evidence_ledger")) for a in answered_actions)
        ledger_entries = sum(len(a.get("evidence_ledger", [])) for a in answered_actions)
        return {"n": n, "correct": f"{correct}/{n}", "abstentions": sum(d["abstained"] for d in diagnostics),
                "wrong_non_abstaining": sum(not d["abstained"] and normalize(r["answer"]) != normalize(validation[str(r["id"])]["answer"]) for r, d in zip(rows, diagnostics)),
                "citation_id_validity": round((cited-invalid)/cited, 3) if cited else None,
                "mean_tool_calls": round(sum(r.get("tool_calls", 0) for r in rows)/max(n,1), 2),
                "mean_tokens": round(sum(r.get("total_tokens", 0) for r in rows)/max(n,1), 1),
                "answered_with_ledger": f"{ledger_answers}/{len(answered_actions)}", "ledger_entries": ledger_entries,
                "invalid_action_retries": sum(r.get("invalid_action_retries", 0) for r in rows)}
    control = read_evidence_rows(STRUCTURED_EVIDENCE_CONTROL)
    trial = read_evidence_rows(STRUCTURED_EVIDENCE_OUTPUT)
    if len(control) != 100 or len(trial) != 100 or [str(r["id"]) for r in control] != [str(r["id"]) for r in trial]:
        raise ValueError("Both full-validation files must contain the same ordered 100 question IDs.")
    print("current_best:", summarize_evidence(control))
    print("structured_evidence:", summarize_evidence(trial))
else:
    print("Run the structured-evidence full validation and keep the current best result before comparing.")
